In [1]:
import sys
import os

# Add the project root to the path
if os.path.abspath('.') not in sys.path:
    sys.path.append(os.path.abspath('.'))

# Import the setup module
from notebook_setup import setup_django
setup_django()

# Now you can import Django models
from common.models import Brokers, Prices, FX, Assets
from users.models import CustomUser
from datetime import date, datetime, timedelta

from tinkoff.invest import Client
from users.models import TinkoffApiToken

user = CustomUser.objects.get(id=1)

TM = TinkoffApiToken.objects.get(user=user, token_type='read_only', sandbox_mode=False, is_active=True)
token = TM.get_token(user=user)
print(f"Token exists: {token is not None}")

Django initialized successfully!
Token exists: True


In [ ]:
selected_brokers = broker_ids = [2]
effective_current_date = date.today()
currency_target = 'USD'
number_of_digits = 2

CustomUser.objects.get(id=1)

In [ ]:
security_id = 7
s = Assets.objects.get(id=security_id)
s.price_at_date(effective_current_date).price

# for field in FX._meta.get_fields():
#     print(field.name)

quote = s.prices.filter(date__lte=effective_current_date).order_by('-date').first()
print(currency_target)
# if currency_target is not None:
#     quote.price = quote.price * FX.get_rate(security.currency, currency_target, effective_current_date, investor=2)['FX']
# quote

FX.get_rate('USD', 'GBP', effective_current_date)

In [ ]:
from django.db.models import Count

duplicates = (
            Prices.objects.values('date', 'security', 'price')
            .annotate(count_id=Count('id'))
            .filter(count_id__gt=1)
        )

total_duplicates = len(duplicates)
print((f"Found {total_duplicates} duplicate entries."))

for i, duplicate in enumerate(duplicates, start=1):
            # Keep one entry and delete the rest
            entries = Prices.objects.filter(
                date=duplicate['date'],
                security=duplicate['security'],
                price=duplicate['price']
            )
            entries_to_keep = entries.first()
            entries.exclude(id=entries_to_keep.id).delete()

            # Print status update
            print(f"Duplicates are being removed: {i} out of {total_duplicates}\r", end='', flush=True)
            
print("\nDuplicates removed.")


In [ ]:
import datetime
from decimal import Decimal

from portfolio_management.core.portfolio_utils import NAV_at_date


def end_of_year_price_correction(user, year, broker_name, target_nav, asset_name):
    
    target_nav = round(Decimal(target_nav), 2)
    
    # Get the broker
    try:
        broker = Brokers.objects.get(name=broker_name)
    except Brokers.DoesNotExist:
        return {"error": f"Broker {broker_name} does not exist."}

    # Get the asset
    try:
        asset = Assets.objects.get(name=asset_name)
    except Assets.DoesNotExist:
        return {"error": f"Asset {asset_name} does not exist."}

    # Calculate end of year date
    end_of_year_date = datetime.date(year, 12, 31)

    # Fetch NAV at the end of the year
    # nav_at_end_of_year = get_nav_at_date(broker, end_of_year_date)
    nav_at_end_of_year = NAV_at_date(user.id, [broker.id], end_of_year_date, asset.currency, [])['Total NAV']
    if nav_at_end_of_year is None:
        return {"error": f"No NAV found for broker {broker_name} at the end of {year}."}

    # Fetch asset price and position at the end of the year
    price_at_end_of_year = asset.price_at_date(end_of_year_date)
    if not price_at_end_of_year:
        return {"error": f"No price found for asset {asset_name} at the end of {year}."}

    position_at_end_of_year = asset.position(end_of_year_date, user, [broker.id])

    # Calculate new price
    old_price = price_at_end_of_year.price
    new_price = old_price + ((target_nav - nav_at_end_of_year) / position_at_end_of_year)

    # Display information
    old_asset_value = round(Decimal(old_price * position_at_end_of_year), 2)
    new_asset_value = round(Decimal(new_price * position_at_end_of_year), 2)

    result = {
        "old_price": old_price,
        "new_price": new_price,
        "old_asset_value": old_asset_value,
        "new_asset_value": new_asset_value,
        "old_nav": nav_at_end_of_year,
        "target_nav": target_nav
    }

    print(f"Old Price: {result['old_price']}, New Price: {result['new_price']}")
    print(f"Old Asset Value: {result['old_asset_value']}, New Asset Value: {result['new_asset_value']}")
    print(f"Old NAV: {result['old_nav']}, Target NAV: {result['target_nav']}")

    # Ask for confirmation
    confirm = input("Do you want to update the price? (yes/no): ")

    if confirm.lower() == 'yes':
        # Update the price
        price_instance, created = Prices.objects.get_or_create(
            security=asset, date=end_of_year_date,
            defaults={'price': new_price}
        )
        if not created:
            price_instance.price = new_price
            price_instance.save()

        result["status"] = "Price updated successfully."
        print("New price is:", price_instance.price)
    else:
        result["status"] = "Price update canceled."

    return result

user = CustomUser.objects.filter(username='Yaroslav').first()
year = 2013
broker_name = 'UBS Pension'
target_nav = 29444.38
asset_name = 'Emerging Markets Equity Fund'

end_of_year_price_correction(user, year, broker_name, target_nav, asset_name)

In [ ]:
from common.models import Accounts

user = CustomUser.objects.get(username='Y')

a = Accounts.objects.get(broker__investor=user, name='Test broker')
a.transactions.filter(quantity__isnull=False)

s = Assets.objects.filter(transactions__broker_account=a, transactions__date__lte=date(2020,6,1))
s.distinct().count()


In [ ]:
async def async_generator():
    yield "Initial async value from generator"
    # Process the received value
    print(f"Sent initial value")
    
    for i in range(3):    
        
        response =yield f'Sending {i}'

        print(f"Response from consumer: {response}")

async def async_consumer():
    gen = async_generator()
    
    # Start the async generator
    result = await gen.asend(None)  # Pass None on the first call
    print(f"Consumer received: {result}")

    async for generator_tick in gen:
        print(f"Received from generator: {generator_tick}")
    
        await gen.asend(f"Message from consumer")
        # print(f"Consumer received: {response}")
    
    # Close the async generator
    await gen.aclose()


await async_consumer()

In [ ]:
from tinkoff.invest import Client

from users.models import TinkoffApiToken

user = CustomUser.objects.get(id=1)

TM = TinkoffApiToken.objects.get(user=user, token_type='read_only', sandbox_mode=False, is_active=True)
token = TM.get_token(user=user)

with Client(token) as client:
    accounts = client.users.get_accounts()
    print(accounts)


In [ ]:
from tinkoff.invest import Client, OperationState, OperationType, GetOperationsByCursorRequest, InstrumentType
from tinkoff.invest.utils import now, quotation_to_decimal
from datetime import timedelta

from constants import OPERATION_TYPE_DESCRIPTIONS, INSTRUMENT_KIND_DESCRIPTIONS
from users.models import TinkoffApiToken

user = CustomUser.objects.get(id=1)

def get_recent_operations(user, from_date=None, to_date=None, days_back=None):
    # Fixed token retrieval method using TinkoffApiToken directly
    TM = TinkoffApiToken.objects.get(user=user, token_type='read_only', sandbox_mode=False, is_active=True)
    token = TM.get_token(user=user)
    
    with Client(token) as client:
        accounts = client.users.get_accounts()
        if not accounts.accounts:
            print("No accounts found.")
            return
        
        # If there's more than one account, let the user choose
        if len(accounts.accounts) > 1:
            print("Available accounts:")
            for i, account in enumerate(accounts.accounts, 1):
                print(f"{i}. {account.name} (ID: {account.id})")
            
            while True:
                try:
                    choice = int(input("Select an account number: ")) - 1
                    if 0 <= choice < len(accounts.accounts):
                        selected_account = accounts.accounts[choice]
                        break
                    else:
                        print("Invalid selection. Please try again.")
                except ValueError:
                    print("Please enter a valid number.")
        else:
            selected_account = accounts.accounts[0]
        
        print(f"\nFetching operations for account: {selected_account.name}")
        
        # Set the time range for operations
        if from_date is None and to_date is None:
            if days_back is not None:
                to_date = now()
                from_date = to_date - timedelta(days=days_back)
            else:
                raise ValueError("days_back must be provided if from_date and to_date are not provided")
        elif from_date is not None:
            if to_date is None:
                to_date = now()
        elif from_date is None:
            if days_back is not None:
                from_date = to_date - timedelta(days=days_back)
            else:
                raise ValueError("days_back must be provided if from_date is not provided")
        
        # Initialize cursor and operations list
        cursor = ""
        all_operations = []
        
        while True:
            # Request operations
            response = client.operations.get_operations_by_cursor(
                GetOperationsByCursorRequest(
                    account_id=selected_account.id,
                    from_=from_date,
                    to=to_date,
                    cursor=cursor,
                    limit=1000,  # Adjust this value as needed
                    operation_types=[],  # Empty list means all types
                    state=OperationState.OPERATION_STATE_EXECUTED
                )
            )
            
            all_operations.extend(response.items)
            
            if not response.has_next:
                break
            
            cursor = response.next_cursor
        
        # Process and display operations
        if not all_operations:
            print("No operations found in the specified time range.")
            return
        
        print(f"\nRecent {len(all_operations)} operations (last {days_back} days):")
        for op in all_operations:
            operation_date = op.date.strftime("%Y-%m-%d %H:%M:%S")
            operation_type = OPERATION_TYPE_DESCRIPTIONS.get(op.type, "Unknown Operation Type")
            instrument_name = op.name
            instrument_kind = INSTRUMENT_KIND_DESCRIPTIONS.get(op.instrument_kind, "Unknown Instrument Type")
            quantity = op.quantity
            price = quotation_to_decimal(op.price) if op.price else "N/A"
            currency = op.payment.currency if op.payment else "N/A"
            payment = quotation_to_decimal(op.payment) if op.payment else "N/A"

            # if operation_type == 'Buy Securities':
            print("-------------")
            print(op)
            print("-------------")
            
            # print(f"Date: {operation_date}")
            # print(f"Type: {operation_type}")
            # print(f"Instrument: {instrument_name}")
            # print(f"Description: {op.description}")
            # print(f"Instrument type: {op.instrument_type}")
            # print(f"Instrument kind: {instrument_kind}")
            # print(f"Quantity: {quantity}")
            # print(f"Price: {price} {currency}")
            # print(f"Total Payment: {payment} {currency}")
            # print(f"Trades list: {op.trades_info}")
            # print("---")

# Example usage:
get_recent_operations(user, from_date=datetime(2023, 12, 1), to_date=datetime(2023, 12, 25)) 

In [ ]:
instrument_uid = '8deb668a-ec57-4fd6-958e-ca5bcaaf2a2d'
figi='TCSS0A102SG9'
position_uid = 'e656cdb3-30b5-44ee-bf24-b3e21d2cb10c'
name = 'Доллар США'
ticker = 'RU000A105QW3'


instrument_uid = '2d882f92-78b4-4904-a030-8cad096235cd' # Vangard

with Client(token) as client:
    try:
        etf = client.instruments.etf_by(
            id_type=3, id=instrument_uid
        )
        print(etf)
    except Exception as e:
        print(f"\n{e}")

    try:
        bonds = client.instruments.bond_by(
            id_type=2, id=ticker, class_code='TQCB'
        )
        # bonds = client.instruments.bond_by(
        #     id_type=3, id='f8c85002-3981-47c6-a9e6-53b730152e9d'
        # )
        print("\n", bonds)
    except Exception as e:
        print(f"\n{e}")
    
    try:
        instruments = client.instruments.get_instrument_by(
            id_type=3, id=instrument_uid
        )
        print("\n", instruments)
    except Exception as e:
        print(f"\n{e}")
    try:
        instruments = client.instruments.get_instrument_by(
            id_type=1, id=figi
        )
        print(f"\n{instruments}")
    except Exception as e:
        print(f"\n{e}")
    try:
        instruments = client.instruments.get_instrument_by(
            id_type=4, id=position_uid
        )
        print(f"\n{instruments}")
    except Exception as e:
        print(f"\n{e}")
    # etf = client.instruments.etf_by(
    #     id_type=4, id=position_uid
    # )
    # asset = client.instruments.get_asset_by(
    #     id=instrument_uid
    # )
    instruments = client.instruments.find_instrument(
        query=instrument_uid
    )
    print(f"\n{instruments}")
    # print(instrument)
    # print(etf)
    # print(asset)
    # print(instrument)
    
    

In [ ]:
from decimal import Decimal

def price_at_date(security, price_date, currency=None):
    print(f"Fetching price for {security.name} as of {price_date} in currency {currency}")
    # Convert date to timezone-aware datetime for query
    quote = security.prices.filter(date__lte=price_date).order_by("-date").first()
    query_date = price_date
    if quote is None:
        # If no quote is found, take the price from the last transaction
        last_transaction = (
            security.transactions.filter(date__lte=query_date, quantity__isnull=False)
            .order_by("-date")
            .first()
        )
        if last_transaction:
            print(
                f"Using last transaction price for {security.name} as of {last_transaction.date}"
            )
            quote = type(
                "obj",
                (object,),
                {"price": last_transaction.price, "date": last_transaction.date},
            )
        else:
            print(f"No transaction found for {security.name} as of {query_date}")
            return None

    if currency is not None:
        if security.is_bond:
            fx_rate = 1
        else:
            fx_rate = FX.get_rate(security.currency, currency, price_date)["FX"]
        print(
            f"Converting price from {security.currency} to {currency} using FX rate {fx_rate}"
        )
        quote.price = quote.price * fx_rate
    print(
        f"Price for {security.name} as of {quote.date} is {quote.price} "
        f"in currency {currency or security.currency}"
    )
    return quote

def calculate_value_at_date(security, date, investor, currency=None, account_ids=None):
    position = security.position(date, investor, account_ids)
    if position == 0:
        return Decimal(0)

    price_quote = price_at_date(security, date, currency)
    if price_quote is None:
        print(f"No price found for {security.name} at {date}")
        return Decimal(0)

    price = price_quote.price  # For bonds: percentage of par

    # For bonds: value = position * (price% / 100) * notional
    # For others: value = position * price
    if security.is_bond:
        effective_notional = security.get_effective_notional(date, investor, account_ids, currency)
        value = position * price * effective_notional / Decimal(100)
        print(
            f"Bond value calculation for {security.name}: "
            f"position={position}, price%={price}, notional={effective_notional}, "
            f"value={value}"
        )
    else:
        value = position * price
        print(
            f"Standard value calculation for {security.name}: "
            f"position={position}, price={price}, value={value}"
        )

    return value

i = CustomUser.objects.get(pk=1)
s = Assets.objects.get(id=137)

calculate_value_at_date(s, "2025-06-01", i, currency="USD")
print("\n")
calculate_value_at_date(s, "2025-06-01", i, currency="RUB")

Fetching price for ХК Новотранс 001P-02 as of 2021-06-01 in currency USD
Using last transaction price for ХК Новотранс 001P-02 as of 2021-04-22 00:00:00+00:00
Converting price from RUB to USD using FX rate 1
Price for ХК Новотранс 001P-02 as of 2021-04-22 00:00:00+00:00 is 100.000000000 in currency USD
Bond value calculation for ХК Новотранс 001P-02: position=50.000000, price%=100.000000000, notional=1000.00, value=50000.00000000000000000


Fetching price for ХК Новотранс 001P-02 as of 2021-06-01 in currency RUB
Using last transaction price for ХК Новотранс 001P-02 as of 2021-04-22 00:00:00+00:00
Converting price from RUB to RUB using FX rate 1
Price for ХК Новотранс 001P-02 as of 2021-04-22 00:00:00+00:00 is 100.000000000 in currency RUB
Bond value calculation for ХК Новотранс 001P-02: position=50.000000, price%=100.000000000, notional=1000.00, value=50000.00000000000000000


Decimal('50000.00000000000000000')

In [4]:
a = Assets.objects.get(id=140)
print(a)
a.bond_metadata.get_current_aci(date(2024,5,5))

CBOM Finance P.L.C. 7.5


{'aci_amount': Decimal('6.15'),
 'aci_days': 30,
 'total_days': 183,
 'coupon_start': datetime.date(2024, 4, 5),
 'coupon_end': datetime.date(2024, 10, 5),
 'next_payment': datetime.date(2024, 10, 5),
 'currency': 'USD'}